# Tiny 1-D CNN on Raw EEG (PyTorch Edition)

**Goal** Train a compact 1-D Convolutional Neural Network (~15 k parameters) in **PyTorch** to classify sleep stages from 30-second raw EEG windows, then compare its performance to the Logistic‐Regression baseline from Notebook 2.

**Workflow**

1. Load the PSG and hypnogram, slicing the data into 30-second epochs.  
2. Pre-process: resample to 100 Hz and z-score each epoch.  
3. Build PyTorch `Dataset` and `DataLoader` objects.  
4. Define a tiny 1-D CNN, train it with early stopping.  
5. Evaluate on a held-out validation set and visualise the confusion matrix.

## 1 – Imports

In [ ]:
# Uncomment the next line if you have to install dependencies
# %pip install torch torchinfo

import mne
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    f1_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

## 2 – Load PSG & attach hypnogram

In [ ]:
PSG_PATH = "./data/ST7011J0-PSG.edf"
HYP_PATH = "./data/ST7011JP-Hypnogram.edf"

raw = mne.io.read_raw_edf(PSG_PATH, preload=True, verbose="error")
raw.pick("eeg")                                # keep Fpz-Cz & Pz-Oz
raw.set_annotations(mne.read_annotations(HYP_PATH))

# Downsample to 100 Hz (Sleep-EDF EEG native = 100 Hz already, but keep generic)
raw.resample(100, npad="auto")
print(raw)

## 3 – Epoch 30-s windows & label mapping

In [ ]:
stage_map = {
    "Sleep stage W": 0,
    "Sleep stage 0": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
}
events, event_id = mne.events_from_annotations(
    raw, event_id=stage_map, chunk_duration=30.0
)
mask = np.isin(events[:, 2], list(event_id.values()))
events = events[mask]

epochs = mne.Epochs(
    raw,
    events,
    event_id=event_id,
    tmin=0.0,
    tmax=30.0 - 1.0 / raw.info["sfreq"],
    baseline=None,
    preload=True,
    on_missing="ignore",
)
print(epochs)

## 4 – Convert to numpy & z-score per epoch
Shapes:
* **X** → (n_epochs, timesteps, n_channels)
* **y** → integer labels {0…4}

In [ ]:
X = epochs.get_data()                          # (n_epochs, n_ch, n_times)
X = np.transpose(X, (0, 2, 1))                 # (n_epochs, time, ch)
n_epochs, n_times, n_ch = X.shape
print("Tensor shape:", X.shape)

# Z-score each epoch across time for each channel
X = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

y = epochs.events[:, 2]
label_names = ["W", "N1", "N2", "N3", "REM"]
print("Class distribution:", np.bincount(y))

## 5 – Train / validation split (stratified)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "Val:", X_val.shape)

## 6 – Dataset & DataLoader

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32
train_dl = DataLoader(EEGDataset(X_train, y_train), batch_size, shuffle=True)
val_dl   = DataLoader(EEGDataset(X_val,   y_val),   batch_size, shuffle=False)

## 7 – Tiny 1-D CNN architecture

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, n_ch, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_ch, 32, kernel_size=7, padding="same"),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding="same"),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)   # (batch, ch, time)
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyCNN(n_ch=X.shape[2], n_classes=len(label_names)).to(device)
summary(model, input_size=(batch_size, X.shape[1], X.shape[2]))

## 8 – Train with early stopping

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
patience, best_val, wait = 10, np.inf, 0
n_epochs = 60
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, n_epochs + 1):
    # ---------- train ----------
    model.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(yb)
    train_loss = running / len(train_dl.dataset)

    # ---------- val ----------
    model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            running += criterion(model(xb), yb).item() * len(yb)
    val_loss = running / len(val_dl.dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    # early stopping
    if val_loss < best_val:
        best_val, wait = val_loss, 0
        best_state = model.state_dict()
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping")
            break

model.load_state_dict(best_state)

## 9 – Validation set evaluation

In [ ]:
model.eval()
with torch.no_grad():
    preds = []
    for xb, _ in val_dl:
        xb = xb.to(device)
        preds.append(model(xb).argmax(1).cpu())
    val_preds = torch.cat(preds).numpy()

print(classification_report(y_val, val_preds, target_names=label_names))

cm = confusion_matrix(y_val, val_preds, labels=range(len(label_names)))
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(6,5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("PyTorch CNN – confusion matrix (validation)")
plt.show()

In [ ]:
cnn_acc = accuracy_score(y_val, val_preds)
cnn_f1  = f1_score(y_val, val_preds, average="macro")

print(f"Validation accuracy (PyTorch CNN): {cnn_acc:.3f}")
print(f"Validation macro-F1 (PyTorch CNN): {cnn_f1:.3f}")

## 10 – Comparison to Logistic Regression baseline

| Metric            | Logistic Regression (Notebook 2) | Tiny CNN (PyTorch) |
|-------------------|----------------------------------|--------------------|
| Accuracy (val)    | ~0.68                            | ~0.72 |
| Macro F1 (val)    | ~0.64                            | ~0.54 |

## 11 – Wrap-up

* The PyTorch model mirrors the TensorFlow/Keras version and stays under **0.5 MB** (~15 k parameters).  
* Training completes in under two minutes on CPU—faster with a GPU.  
* Possible extensions:  
  * Data augmentation (additive noise, temporal shifts).  
  * Adding EOG/EMG channels for richer multi-modal input.  
  * Pre-training on the entire Sleep-EDF cohort before subject-specific fine-tuning.